# Python Classes

> 📘 **Python Mastery** · Module 08 — Object-Oriented Programming (OOP) · Lesson 1/5

Until now your data lived in variables and your logic lived in functions, in two separate worlds. A **class** bundles them together into one blueprint — and nearly every library you will ever import (including every machine-learning framework) hands you classes to use.

## 🎯 Learning Objectives

- Explain what problems object-oriented programming (OOP) solves compared to loose functions and lists
- Define a class using the `class` keyword, PascalCase naming, and a docstring
- Initialize objects with the `__init__` constructor and instance attributes
- Describe exactly what `self` is and show that Python passes it automatically
- Contrast **class attributes** (shared) with **instance attributes** (per object)
- Customize how objects print using `__str__` and `__repr__`

## 1. Why Classes? From Procedural to OOP

Procedural code keeps **data** (variables) and **behavior** (functions) apart, and *you* must remember which belongs to which. A class glues them together: the data becomes **attributes**, the functions become **methods**, and both travel inside one object. Think of a class as a blank form — every filled-in copy is an object.

**Syntax:**

```python
class ClassName:
    """What this blueprint represents."""

    def __init__(self, param1, param2):
        self.param1 = param1      # attribute: data attached to each object
        self.param2 = param2

    def do_something(self):       # method: behavior of each object
        ...uses self.param1...
```

**Example:** the same student record, written both ways.

In [1]:
# Procedural style: data and behavior live in separate worlds
student_name = "Sarah"
student_grades = [88, 92, 79]

def average_of(grades):
    return sum(grades) / len(grades)

def has_passed(grades):
    return average_of(grades) >= 60

print(f"{student_name}: avg={average_of(student_grades)}, passed={has_passed(student_grades)}")

# Problem: NOTHING ties student_name to student_grades except discipline.
# With 300 students you juggle parallel lists and pray the indexes stay aligned.

Sarah: avg=86.33333333333333, passed=True


In [2]:
# OOP style: the data AND the operations on it travel together
class Student:
    def __init__(self, name, grades):
        self.name = name          # attribute: this student's data
        self.grades = grades

    def average(self):            # method: this student's behavior
        return sum(self.grades) / len(self.grades)

    def has_passed(self):
        return self.average() >= 60

sarah = Student("Sarah", [88, 92, 79])
rafi = Student("Rafi", [55, 61, 47])

print(sarah.average(), sarah.has_passed())
print(rafi.average(), rafi.has_passed())

86.33333333333333 True
54.333333333333336 False


## 2. Defining a Class: the `class` Keyword

One line — `class Name:` — creates a new *type* and gives it a name. By convention class names use **PascalCase** (`BankAccount`, not `bank_account`). A class with nothing inside is legal: use the `pass` placeholder, and give every serious class a docstring saying what it models.

**Syntax:**

```python
class Empty:                 # legal, useful as a placeholder or marker
    pass


class Filled:
    """Docstring describes the blueprint."""
    def __init__(self):
        ...
```

**Example:** an empty class is still a working type.

In [3]:
class Point:
    """A location on a 2D grid. (We will fill it in later lessons.)"""
    pass                      # 'pass' = "nothing here yet"; keeps the class valid

blank = Point()               # even an empty class can be instantiated
print(Point)                  # the class is itself an object
print(type(Point))            # a class is an object of type... 'type'
print(blank)                  # a fresh, blank instance

<class '__main__.Point'>
<class 'type'>


## 3. `__init__`: the Constructor

`__init__` is a special method Python calls **automatically** the moment you create an object. Its job is to receive starting values and attach them to the new object as **instance attributes** (`self.x = x`). You never call `__init__` yourself, and it must not return anything — creating the object already produces the value.

**Syntax:**

```python
class Dog:
    def __init__(self, name, age):
        self.name = name      # instance attribute
        self.age = age

rex = Dog("Rex", 3)           # -> Python calls Dog.__init__(rex, "Rex", 3)
```

**Example:** watch the constructor fire twice.

In [4]:
class Dog:
    """A pet dog identified by name and age."""

    def __init__(self, name, age):
        print(f"-> __init__ fired: building a dog named {name}")
        self.name = name      # attach data to THIS object
        self.age = age

rex = Dog("Rex", 3)           # Dog("Rex", 3) triggers __init__ for you
bella = Dog("Bella", 5)

print(rex.name, rex.age)
print(bella.name, bella.age)

-> __init__ fired: building a dog named Rex
-> __init__ fired: building a dog named Bella
Rex 3
Bella 5


## 4. 🔍 Understanding `self`

`self` is simply **the object the method was called on** — nothing mystical. When you write `rex.bark()`, Python quietly turns it into `Dog.bark(rex)`, handing the function the instance as its first argument. The name `self` is a rock-solid community convention (technically not a keyword), and you should never break it.

> 🔍 **Under the Hood:** Methods are ordinary function objects stored in the class's namespace. Evaluating `rex.bark` does not return the raw function — it builds a *bound method*: a small wrapper that pairs the function with `rex`. Calling the wrapper injects `rex` as the first parameter. This is also exactly why forgetting `self` produces the famous `TypeError: bark() takes 0 positional arguments but 1 was given` — Python tried to hand your method the instance, and the method refused it.

**Syntax:**

```python
class C:
    def method(self, other_args):    # self = the instance, always first
        ...

obj.method(x)      # sugar for: C.method(obj, x)
```

**Example:** prove both call forms are identical.

In [5]:
class Dog:
    def __init__(self, name):
        self.name = name

    def bark(self):
        return f"{self.name} says Woof!"

rex = Dog("Rex")

# These two lines are EXACTLY equivalent:
print(rex.bark())          # the sugar humans write
print(Dog.bark(rex))       # what Python actually executes: rex becomes self

print(rex.bark() == Dog.bark(rex))

Rex says Woof!
Rex says Woof!
True


In [6]:
class SelfCheck:
    def who_am_i(self):
        return self               # the method receives the instance itself

obj = SelfCheck()

print(obj.who_am_i() is obj)      # True -- 'self' IS the object, not a copy

# 'self' is convention, not a keyword (renaming is legal and forbidden):
class Silly:
    def greet(this_one):          # works... and horrifies every reviewer
        return f"Hi from {this_one.__class__.__name__}"

print(Silly().greet())

True
Hi from Silly


## 5. Instance Attributes and Instance Methods

An **instance attribute** is a variable attached to one particular object (`self.owner`); an **instance method** is a function defined on the class but operating on one object through `self`. Together they give each object its own state *and* its own skills.

**Syntax:**

```python
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner        # attribute (data)
        self.balance = balance

    def deposit(self, amount):    # method (behavior)
        self.balance += amount
```

**Example:** our `BankAccount` — it returns for Lessons 2 and 5.

In [7]:
class BankAccount:
    """A simple bank account: state + behavior bundled."""

    def __init__(self, owner, balance=0):
        self.owner = owner        # instance attribute
        self.balance = balance    # instance attribute

    def deposit(self, amount):    # instance method
        self.balance += amount
        return f"{self.owner}: deposited {amount}, balance {self.balance}"

    def withdraw(self, amount):
        if amount > self.balance:
            return f"{self.owner}: insufficient funds (have {self.balance})"
        self.balance -= amount
        return f"{self.owner}: withdrew {amount}, balance {self.balance}"

acc = BankAccount("Sarah", 100)
print(acc.deposit(50))
print(acc.withdraw(30))
print(acc.withdraw(1000))

Sarah: deposited 50, balance 150
Sarah: withdrew 30, balance 120
Sarah: insufficient funds (have 120)


## 6. Class Attributes vs Instance Attributes

Attributes written **directly in the class body** belong to the class, so every instance sees the *same* value — ideal for constants and counters. Attributes created as `self.x` inside `__init__` belong to each individual **instance**. Analogy: a class attribute is a whiteboard bolted to the classroom wall; instance attributes are the notebooks students carry.

**Syntax:**

```python
class Dog:
    species = "Canis familiaris"     # CLASS attribute: shared by all dogs

    def __init__(self, name):
        self.name = name             # INSTANCE attribute: unique per dog
```

**Example:** a counter shared by every account ever opened.

In [8]:
class BankAccount:
    bank_name = "Python National Bank"   # CLASS attribute: one shared value
    total_accounts = 0                   # shared counter across ALL instances

    def __init__(self, owner):
        self.owner = owner               # INSTANCE attribute: unique per account
        BankAccount.total_accounts += 1  # bump the shared counter on creation

a = BankAccount("Sarah")
b = BankAccount("Rafi")
c = BankAccount("Amina")

print(BankAccount.total_accounts)        # 3 -- read straight from the class
print(a.total_accounts, b.total_accounts, c.total_accounts)  # instances see it too

3
3 3 3


In [9]:
BankAccount.bank_name = "PNB Global"     # change ONE class attribute...
print(a.bank_name, "|", b.bank_name)     # ...and every instance reflects it

# (Writing a.bank_name = ... would instead create an instance-level copy
#  that shadows the class one -- Lesson 2 covers that trick.)

PNB Global | PNB Global


## 7. `__str__` and `__repr__`: Controlling Printing

Printing a fresh object shows `<__main__.Dog object at 0x...>` — useless. Define `__str__` for a friendly, user-facing description (used by `print()` and `str()`), and `__repr__` for an unambiguous, developer-facing one (used in the interactive console and inside lists/dicts). Rule of thumb: `__repr__` should look like the code that would rebuild the object. If `__str__` is missing, Python falls back to `__repr__`.

**Syntax:**

```python
class Book:
    def __str__(self):                 # friendly: for end users
        return "readable description"
    def __repr__(self):                # precise: for developers/debugging
        return "Book(...)"
```

**Example:** one object, two costumes.

In [10]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):                 # friendly: for end users / print()
        return f"'{self.title}' ({self.pages} pages)"

    def __repr__(self):                # precise: for developers / debugging
        return f"Book(title={self.title!r}, pages={self.pages})"

book = Book("Fluent Python", 792)

print(book)            # print() prefers __str__
print(repr(book))      # repr() asks for __repr__
print([book, book])    # containers ALWAYS display their items via __repr__

'Fluent Python' (792 pages)
Book(title='Fluent Python', pages=792)
[Book(title='Fluent Python', pages=792), Book(title='Fluent Python', pages=792)]


In [11]:
class Config:
    def __init__(self, mode):
        self.mode = mode

    def __repr__(self):               # only __repr__ defined
        return f"Config(mode={self.mode!r})"

cfg = Config("training")
print(cfg)                            # no __str__? Python falls back to __repr__

class Bare:
    pass                              # neither defined

print(Bare())                         # default: class name + memory address

Config(mode='training')


## 8. Putting It Together: a Small `Person` Class

Here is our running example family in miniature — `Person` grows a `Student` subclass in Lesson 3, and `BankAccount` gets properly encapsulated in Lesson 5. Notice how every idea from this lesson appears at once: constructor, instance attributes, a shared class attribute, methods, and a readable `__repr__`.

**Example:**

In [12]:
class Person:
    """A human with a name and an age -- our running example."""

    species = "Homo sapiens"          # shared by every person

    def __init__(self, name, age):
        self.name = name
        self.age = age

    def greet(self):
        return f"Hi, I'm {self.name}, age {self.age}."

    def __repr__(self):
        return f"Person(name={self.name!r}, age={self.age})"

sarah = Person("Sarah", 22)
rafi = Person("Rafi", 19)

print(sarah.greet())
print(rafi.greet())
print(sarah.species, "/", rafi.species)   # one shared class attribute
print([sarah, rafi])                      # readable thanks to __repr__

Hi, I'm Sarah, age 22.
Hi, I'm Rafi, age 19.
Homo sapiens / Homo sapiens
[Person(name='Sarah', age=22), Person(name='Rafi', age=19)]


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Forgetting `self` in a method definition | `TypeError: ...takes 0 positional arguments but 1 was given` on every call | Make `self` the first parameter of every instance method |
| Writing `def __init__(self, tags=[])` | One shared list mutated by **all** instances | Use `tags=None`, then `self.tags = tags if tags is not None else []` |
| Naming classes like variables (`bank_account`) | Breaks the ecosystem-wide convention and confuses readers | Use PascalCase: `BankAccount` |
| Initializing state in a separate `setup()` method | Forgettable — half-initialized objects roam the program | Do all essential initialization in `__init__` |
| Putting `print()` in methods instead of `return` | Results can't be reused, tested, or chained | Return values; print at the call site |

## 💡 Best Practices & Pro Tips

- Give every class a one-line docstring: six words now save an hour of archaeology later.
- One class, one job. If your class description contains the word "and", consider splitting it.
- Prefer methods that **return** answers over methods that print — callers stay in control.
- Reach for a class attribute only for genuine constants and counters; per-object state belongs in `self`.
- Keep a helpful `__repr__` on data classes while developing: debugging a list of objects becomes a pleasure.
- 🤖 **AI-engineering relevance:** ML code is class-shaped end to end — scikit-learn estimators (`model = LinearRegression()` then `model.fit(X, y)`), PyTorch `nn.Module` subclasses, Keras layers. Being able to parse `__init__`, `self`, and attributes is the entry ticket to reading any model source code.

## 📌 Summary

| Piece | What it does | Example |
|---|---|---|
| `class Name:` | Declares a blueprint (new type) | `class Dog:` |
| `__init__(self, ...)` | Runs automatically at creation; sets starting state | `def __init__(self, name): self.name = name` |
| `self` | The instance the method acts on — passed automatically | `rex.bark()` → `Dog.bark(rex)` |
| `self.attr = value` | Creates an instance attribute | `self.age = 3` |
| `Class.attr` | Class attribute, shared by all instances | `Dog.species = "Canis familiaris"` |
| `__str__(self)` | Friendly text, used by `print()` | `return f"{self.name} ({self.age})"` |
| `__repr__(self)` | Unambiguous text for developers and containers | `return f"Dog({self.name!r})"` |
| `pass` | Placeholder body for an empty class | `class Stub: pass` |

**Key takeaways**
- A class bundles data + behavior; an object is one filled-in copy of that blueprint.
- `self` **is** the instance — Python supplies it automatically on every `obj.method()` call.
- Class attributes are shared by everyone; instance attributes are private property of each object.
- `__repr__` is for developers, `__str__` is for users — define `__repr__` at minimum.

> 🔗 **Next Lesson:** [02 · Python Objects](../02_Objects/) — stamp out many objects from one class and learn how they live, breathe, and get identified in memory.